# 신용잔고 & 대주잔고 시각화
- 데이터: `신용잔고_대주잔고_20230102_20251231.csv`
- 컬럼: date, ticker, name, 신용잔고주, 신용잔고금액, 신용비율, 대주잔고주, 대주잔고금액, 대주비율

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('신용잔고_대주잔고_20230102_20251231.csv', encoding='utf-8')
df['date'] = pd.to_datetime(df['date'])
print(f'데이터 shape: {df.shape}')
print(f'기간: {df["date"].min()} ~ {df["date"].max()}')
print(f'종목 수: {df["ticker"].nunique()}')
df.head()

데이터 shape: (207394, 9)
기간: 2023-01-02 00:00:00 ~ 2025-12-30 00:00:00
종목 수: 287


,date,ticker,name,신용잔고주,신용잔고금액,신용비율,대주잔고주,대주잔고금액,대주비율
0,2023-01-02,A005930,삼성전자,7746774,46579438,0.12,11449,66085,0.0
1,2023-01-03,A005930,삼성전자,7722398,46265249,0.12,10690,61623,0.0
2,2023-01-04,A005930,삼성전자,7438118,44687788,0.12,14019,80370,0.0
3,2023-01-05,A005930,삼성전자,7240979,43669976,0.11,15640,89845,0.0
4,2023-01-06,A005930,삼성전자,6980953,42219915,0.10,19208,111063,0.0


## 1. 전체 시장 신용잔고주 추이

In [2]:
# 일자별 전체 시장 신용잔고주 합계
market_credit = df.groupby('date').agg(
    신용잔고주_합계=('신용잔고주', 'sum'),
    신용잔고금액_합계=('신용잔고금액', 'sum')
).reset_index()

fig = px.line(market_credit, x='date', y='신용잔고주_합계',
              title='전체 시장 신용잔고주 추이',
              labels={'date': '날짜', '신용잔고주_합계': '신용잔고주 (합계)'})
fig.update_layout(template='plotly_white', hovermode='x unified')
fig.show()

## 2. 전체 시장 신용잔고금액 추이

In [3]:
fig = px.line(market_credit, x='date', y='신용잔고금액_합계',
              title='전체 시장 신용잔고금액 추이',
              labels={'date': '날짜', '신용잔고금액_합계': '신용잔고금액 (합계, 백만원)'})
fig.update_layout(template='plotly_white', hovermode='x unified')
fig.show()

## 3. 전체 시장 대주잔고주 추이

In [4]:
market_borrow = df.groupby('date').agg(
    대주잔고주_합계=('대주잔고주', 'sum'),
    대주잔고금액_합계=('대주잔고금액', 'sum')
).reset_index()

fig = px.line(market_borrow, x='date', y='대주잔고주_합계',
              title='전체 시장 대주잔고주 추이',
              labels={'date': '날짜', '대주잔고주_합계': '대주잔고주 (합계)'})
fig.update_layout(template='plotly_white', hovermode='x unified')
fig.show()

## 4. 전체 시장 대주잔고금액 추이

In [5]:
fig = px.line(market_borrow, x='date', y='대주잔고금액_합계',
              title='전체 시장 대주잔고금액 추이',
              labels={'date': '날짜', '대주잔고금액_합계': '대주잔고금액 (합계, 백만원)'})
fig.update_layout(template='plotly_white', hovermode='x unified')
fig.show()

## 5. 전체 시장 평균 신용비율 추이

In [6]:
market_ratio = df.groupby('date').agg(
    신용비율_평균=('신용비율', 'mean'),
    대주비율_평균=('대주비율', 'mean')
).reset_index()

fig = px.line(market_ratio, x='date', y='신용비율_평균',
              title='전체 시장 평균 신용비율 추이',
              labels={'date': '날짜', '신용비율_평균': '신용비율 (평균, %)'})
fig.update_layout(template='plotly_white', hovermode='x unified')
fig.show()

## 6. 전체 시장 평균 대주비율 추이

In [7]:
fig = px.line(market_ratio, x='date', y='대주비율_평균',
              title='전체 시장 평균 대주비율 추이',
              labels={'date': '날짜', '대주비율_평균': '대주비율 (평균, %)'})
fig.update_layout(template='plotly_white', hovermode='x unified')
fig.show()

## 7. 신용잔고 vs 대주잔고 (금액 비교, 서브플롯)

In [8]:
market_all = market_credit.merge(market_borrow, on='date')

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['신용잔고금액 합계', '대주잔고금액 합계'],
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=market_all['date'], y=market_all['신용잔고금액_합계'],
                         mode='lines', name='신용잔고금액', line=dict(color='#EF553B')), row=1, col=1)
fig.add_trace(go.Scatter(x=market_all['date'], y=market_all['대주잔고금액_합계'],
                         mode='lines', name='대주잔고금액', line=dict(color='#636EFA')), row=2, col=1)

fig.update_layout(height=600, title='신용잔고 vs 대주잔고 금액 추이',
                  template='plotly_white', hovermode='x unified')
fig.show()

## 8. 신용비율 vs 대주비율 (이중축)

In [9]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=market_ratio['date'], y=market_ratio['신용비율_평균'],
                         mode='lines', name='신용비율 (평균)', line=dict(color='#EF553B')),
              secondary_y=False)
fig.add_trace(go.Scatter(x=market_ratio['date'], y=market_ratio['대주비율_평균'],
                         mode='lines', name='대주비율 (평균)', line=dict(color='#636EFA')),
              secondary_y=True)

fig.update_layout(title='신용비율 vs 대주비율 추이', template='plotly_white', hovermode='x unified')
fig.update_yaxes(title_text='신용비율 (%)', secondary_y=False)
fig.update_yaxes(title_text='대주비율 (%)', secondary_y=True)
fig.show()

## 9. 상위 10 종목별 신용잔고주 추이

In [10]:
# 최근 날짜 기준 신용잔고주 상위 10종목
latest = df['date'].max()
top10_credit = df[df['date'] == latest].nlargest(10, '신용잔고주')['ticker'].tolist()

df_top10_credit = df[df['ticker'].isin(top10_credit)]

fig = px.line(df_top10_credit, x='date', y='신용잔고주', color='name',
              title='신용잔고주 상위 10종목 추이',
              labels={'date': '날짜', '신용잔고주': '신용잔고주', 'name': '종목명'})
fig.update_layout(template='plotly_white', hovermode='x unified', height=500)
fig.show()

## 10. 상위 10 종목별 대주잔고주 추이

In [11]:
# 최근 날짜 기준 대주잔고주 상위 10종목
top10_borrow = df[df['date'] == latest].nlargest(10, '대주잔고주')['ticker'].tolist()

df_top10_borrow = df[df['ticker'].isin(top10_borrow)]

fig = px.line(df_top10_borrow, x='date', y='대주잔고주', color='name',
              title='대주잔고주 상위 10종목 추이',
              labels={'date': '날짜', '대주잔고주': '대주잔고주', 'name': '종목명'})
fig.update_layout(template='plotly_white', hovermode='x unified', height=500)
fig.show()

## 11. 신용비율 상위 10종목 추이

In [12]:
top10_credit_ratio = df[df['date'] == latest].nlargest(10, '신용비율')['ticker'].tolist()
df_top10_cr = df[df['ticker'].isin(top10_credit_ratio)]

fig = px.line(df_top10_cr, x='date', y='신용비율', color='name',
              title='신용비율 상위 10종목 추이',
              labels={'date': '날짜', '신용비율': '신용비율 (%)', 'name': '종목명'})
fig.update_layout(template='plotly_white', hovermode='x unified', height=500)
fig.show()

## 12. 대주비율 상위 10종목 추이

In [13]:
top10_borrow_ratio = df[df['date'] == latest].nlargest(10, '대주비율')['ticker'].tolist()
df_top10_br = df[df['ticker'].isin(top10_borrow_ratio)]

fig = px.line(df_top10_br, x='date', y='대주비율', color='name',
              title='대주비율 상위 10종목 추이',
              labels={'date': '날짜', '대주비율': '대주비율 (%)', 'name': '종목명'})
fig.update_layout(template='plotly_white', hovermode='x unified', height=500)
fig.show()

## 13. 신용잔고금액 분포 (최근 날짜 기준 히스토그램)

In [14]:
df_latest = df[df['date'] == latest]

fig = px.histogram(df_latest, x='신용잔고금액', nbins=50,
                   title=f'신용잔고금액 분포 ({latest.strftime("%Y-%m-%d")})',
                   labels={'신용잔고금액': '신용잔고금액 (백만원)'})
fig.update_layout(template='plotly_white')
fig.show()

## 14. 대주잔고금액 분포 (최근 날짜 기준 히스토그램)

In [15]:
fig = px.histogram(df_latest, x='대주잔고금액', nbins=50,
                   title=f'대주잔고금액 분포 ({latest.strftime("%Y-%m-%d")})',
                   labels={'대주잔고금액': '대주잔고금액 (백만원)'})
fig.update_layout(template='plotly_white')
fig.show()

## 15. 신용비율 vs 대주비율 산점도 (최근 날짜)

In [16]:
fig = px.scatter(df_latest, x='신용비율', y='대주비율', hover_data=['name', 'ticker'],
                 title=f'신용비율 vs 대주비율 ({latest.strftime("%Y-%m-%d")})',
                 labels={'신용비율': '신용비율 (%)', '대주비율': '대주비율 (%)'},
                 size='신용잔고금액', size_max=20, color='신용잔고금액',
                 color_continuous_scale='Reds')
fig.update_layout(template='plotly_white')
fig.show()

## 16. 개별 종목 조회 (인터랙티브)

In [17]:
# 조회하고 싶은 종목명을 변경하세요
target_name = '삼성전자'

df_target = df[df['name'] == target_name].copy()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=[f'{target_name} - 신용잔고주 & 대주잔고주',
                                    f'{target_name} - 신용잔고금액 & 대주잔고금액',
                                    f'{target_name} - 신용비율 & 대주비율'],
                    vertical_spacing=0.08)

# 잔고주
fig.add_trace(go.Scatter(x=df_target['date'], y=df_target['신용잔고주'],
                         name='신용잔고주', line=dict(color='#EF553B')), row=1, col=1)
fig.add_trace(go.Scatter(x=df_target['date'], y=df_target['대주잔고주'],
                         name='대주잔고주', line=dict(color='#636EFA')), row=1, col=1)

# 잔고금액
fig.add_trace(go.Scatter(x=df_target['date'], y=df_target['신용잔고금액'],
                         name='신용잔고금액', line=dict(color='#EF553B', dash='dash')), row=2, col=1)
fig.add_trace(go.Scatter(x=df_target['date'], y=df_target['대주잔고금액'],
                         name='대주잔고금액', line=dict(color='#636EFA', dash='dash')), row=2, col=1)

# 비율
fig.add_trace(go.Scatter(x=df_target['date'], y=df_target['신용비율'],
                         name='신용비율', line=dict(color='#EF553B', dash='dot')), row=3, col=1)
fig.add_trace(go.Scatter(x=df_target['date'], y=df_target['대주비율'],
                         name='대주비율', line=dict(color='#636EFA', dash='dot')), row=3, col=1)

fig.update_layout(height=900, title=f'{target_name} 신용/대주 잔고 종합',
                  template='plotly_white', hovermode='x unified')
fig.show()